# Storm CNN1D-GRU — Leakage-Free Corrected Notebook

**Purpose:** corrected, leakage-free version of the original capstone model.

**Dataset:** `storm_prediction_dataset.csv`

The held-out test set is isolated before any data-dependent preprocessing.  
For time-series models, chronological splitting is performed **before** scaling and sliding-window construction.  
Validation data are used for early stopping/model selection; the test set is evaluated only after training is complete.

In [ ]:
# Install only if needed in Colab
!pip -q install pandas numpy matplotlib scikit-learn tensorflow

import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
DATA_FILE = "storm_prediction_dataset.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("Please upload", DATA_FILE)
        uploaded = files.upload()
    except Exception as e:
        raise FileNotFoundError(DATA_FILE + " was not found.") from e

assert os.path.exists(DATA_FILE), "Missing file: " + DATA_FILE

In [ ]:
df = pd.read_csv(DATA_FILE)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
invalid_ts = int(df["timestamp"].isna().sum())
df = df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

FEATURES = ["wind_speed_mps", "rain_percent", "water_percent", "distance_cm"]
TARGET = "fused_event"
SEQUENCE_LENGTH = 10

CLASS_NAMES = ["Cyclone+Flood", "CycloneOnly", "FloodLikely", "Normal", "StormOnly"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

unknown = sorted(set(df[TARGET].astype(str)) - set(CLASS_NAMES))
assert not unknown, "Unexpected target labels: " + str(unknown)

df["target_id"] = df[TARGET].map(CLASS_TO_ID).astype(int)

print("Rows after cleaning:", len(df))
print("Invalid timestamps removed:", invalid_ts)
print("Overall class counts:")
print(df[TARGET].value_counts())

In [ ]:
n = len(df)
train_end = int(0.64 * n)
val_end = int(0.80 * n)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

scaler = MinMaxScaler()
scaler.fit(train_df[FEATURES])

def build_sequences(part_df, sequence_length=SEQUENCE_LENGTH):
    X_scaled = scaler.transform(part_df[FEATURES])
    y = part_df["target_id"].to_numpy()

    X_seq, y_seq = [], []
    for i in range(sequence_length, len(part_df)):
        X_seq.append(X_scaled[i-sequence_length:i])
        y_seq.append(y[i])

    return np.asarray(X_seq, dtype=np.float32), np.asarray(y_seq, dtype=np.int32)

X_train, y_train = build_sequences(train_df)
X_val,   y_val   = build_sequences(val_df)
X_test,  y_test  = build_sequences(test_df)

def counts(y):
    vals, c = np.unique(y, return_counts=True)
    return {CLASS_NAMES[int(v)]: int(n) for v, n in zip(vals, c)}

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Train counts:", counts(y_train))
print("Val counts  :", counts(y_val))
print("Test counts :", counts(y_test))

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, GRU, Conv1D, MaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, len(FEATURES))),
    Conv1D(32, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),
    GRU(32, unroll=True, activation="tanh", recurrent_activation="sigmoid"),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(len(CLASS_NAMES), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(prob, axis=1)

acc = accuracy_score(y_test, y_pred)
prec_w = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec_w = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1_w = f1_score(y_test, y_pred, average="weighted", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Accuracy          : {acc:.6f}")
print(f"Weighted precision: {prec_w:.6f}")
print(f"Weighted recall   : {rec_w:.6f}")
print(f"Weighted F1       : {f1_w:.6f}")
print(f"Macro F1          : {f1_macro:.6f}")
print()
print(classification_report(
    y_test, y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=list(range(len(CLASS_NAMES))))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d", xticks_rotation=45)
plt.title("Storm CNN1D-GRU — Leakage-Free Test Set")
plt.show()

pd.DataFrame({
    "true_class": [CLASS_NAMES[i] for i in y_test],
    "predicted_class": [CLASS_NAMES[i] for i in y_pred]
}).to_csv("storm_leakage_free_test_predictions.csv", index=False)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Storm CNN1D-GRU Training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
model.save("storm_fusion_event_leakage_free.keras")

meta = {
    "features": FEATURES,
    "class_names": CLASS_NAMES,
    "sequence_length": SEQUENCE_LENGTH,
    "scaler_min": scaler.min_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "split": {"train": 0.64, "validation": 0.16, "test": 0.20},
    "seed": SEED
}
with open("storm_fusion_event_leakage_free_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter._experimental_lower_tensor_list_ops = True
converter.experimental_enable_resource_variables = False
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("storm_fusion_event_leakage_free.tflite", "wb") as f:
    f.write(tflite_model)

print("Saved corrected storm artifacts.")